# Регрессия IC50

Здесь сравниваются несколько регрессионных моделей для прогноза `IC50, mM` по дескрипторам.

IC50 распределён с тяжёлым хвостом, так что `log1p` для таргета здесь особенно уместен.

In [9]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from types import SimpleNamespace

import pandas as pd
from IPython.display import Image, display

from src.common.config import RESULTS_DIR, TASKS
from src.common.data import find_dataset_path, load_dataset
from src.common.preprocessing import prepare_task_data
from src.common.training import run_regression_si_task, run_supervised_task

## Настройки запуска

По умолчанию стоит полный режим `nested`. Если нужен быстрый прогон для проверки,
достаточно переключить `EVALUATION_STRATEGY` на `"holdout"`.

In [3]:
TASK_NAME = 'regression_ic50'
EVALUATION_STRATEGY = 'nested'
MODELS = None
SKIP_CATBOOST = False
OUTER_FOLDS = 5
INNER_FOLDS = 3
TEST_SIZE = 0.2
RANDOM_SEED = 42
TOP_K_IMPORTANCE = 20

task = TASKS[TASK_NAME]
dataframe = load_dataset(find_dataset_path())
prepared = prepare_task_data(dataframe, task)

print(f'Задача: {task.title}')
print(f'Матрица признаков: {prepared["X"].shape}')
print(f'Число признаков после фильтрации: {len(prepared["feature_columns"])}')
print(f'Статус проверки на утечку: {prepared["leakage_report"]["status"]}')

Задача: Регрессия: IC50
Матрица признаков: (1001, 210)
Число признаков после фильтрации: 210
Статус проверки на утечку: passed


## Быстрый срез по таргету

Перед обучением полезно один раз посмотреть, что именно мы подаём в модель.

In [10]:
display(prepared['y'].describe().to_frame(name=task.target_column).T)

,count,mean,std,min,25%,50%,75%,max
"IC50, mM",1001.0,222.805156,402.169734,0.003517,12.515396,46.585183,224.975928,4128.529377


## Запуск эксперимента

Эта ячейка пересчитывает результаты, пишет артефакты в `results/` и обновляет текстовый отчёт в `reports/`.

In [5]:
args = SimpleNamespace(
    evaluation_strategy=EVALUATION_STRATEGY,
    models=MODELS,
    skip_catboost=SKIP_CATBOOST,
    outer_folds=OUTER_FOLDS,
    inner_folds=INNER_FOLDS,
    test_size=TEST_SIZE,
    random_seed=RANDOM_SEED,
    top_k_importance=TOP_K_IMPORTANCE,
)

summary = run_supervised_task(task, args)
summary

2026-04-20 17:53:15,403 | INFO | Running regression_ic50 with models: ['dummy', 'ridge', 'lasso', 'knn', 'svr', 'random_forest', 'extra_trees', 'gradient_boosting', 'catboost']
2026-04-20 17:53:15,404 | INFO | Evaluating model dummy
2026-04-20 17:53:15,870 | INFO | Evaluating model ridge
2026-04-20 17:53:16,881 | INFO | Evaluating model lasso
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.578e+00, tolerance: 1.856e-01
  model = cd_fast.enet_coordinate_descent(
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to i

{'task_name': 'regression_ic50',
 'title': 'Регрессия: IC50',
 'problem_type': 'regression',
 'target_column': 'IC50, mM',
 'threshold': None,
 'primary_metric': 'mae',
 'evaluation_strategy': 'nested',
 'random_seed': 42,
 'data_contract_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/data_contract.json',
 'leaderboard_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/regression_ic50/leaderboard.csv',
 'winner': {'task_name': 'regression_ic50',
  'problem_type': 'regression',
  'target_column': 'IC50, mM',
  'primary_metric': 'mae',
  'evaluation_strategy': 'nested',
  'model_name': 'knn',
  'mode': 'direct',
  'fit_seconds': 5.902440957957879,
  'best_params_json': '{"n_neighbors": 5, "p": 2, "weights": "uniform"}',
  'mae': 166.81944814433055,
  'mae_std': 29.454376615815047,
  'rmse': 339.40720151490194,
  'rmse_std': 91.84188766615985,
  'r2': 0.2384208445118725,
  'r2_std': 0.08237753032

## Лидерборд

Сравнение разных моделей.

In [6]:
leaderboard = pd.read_csv(RESULTS_DIR / 'regression_ic50' / 'leaderboard.csv')
display(leaderboard)

,task_name,problem_type,target_column,primary_metric,evaluation_strategy,model_name,mode,fit_seconds,best_params_json,mae,mae_std,rmse,rmse_std,r2,r2_std
0,regression_ic50,regression,"IC50, mM",mae,nested,knn,direct,5.902441,"{""n_neighbors"": 5, ""p"": 2, ""weights"": ""uniform""}",1.668194e+02,2.945438e+01,3.394072e+02,9.184189e+01,2.384208e-01,8.237753e-02
1,regression_ic50,regression,"IC50, mM",mae,nested,extra_trees,direct,108.111359,"{""max_depth"": 16, ""max_features"": 0.5, ""min_sa...",1.701811e+02,2.446574e+01,3.444361e+02,6.780566e+01,1.857380e-01,1.471851e-01
2,regression_ic50,regression,"IC50, mM",mae,nested,random_forest,direct,234.741834,"{""max_depth"": null, ""max_features"": 0.5, ""min_...",1.701869e+02,3.003423e+01,3.368097e+02,8.019319e+01,2.391318e-01,7.686421e-02
3,regression_ic50,regression,"IC50, mM",mae,nested,gradient_boosting,direct,232.675917,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",1.722474e+02,3.122120e+01,3.343993e+02,7.920968e+01,2.495675e-01,7.733183e-02
4,regression_ic50,regression,"IC50, mM",mae,nested,catboost,direct,898.236001,"{""depth"": 6, ""iterations"": 400, ""l2_leaf_reg"":...",1.733955e+02,2.747290e+01,3.536454e+02,7.467674e+01,1.292342e-01,2.565705e-01
5,regression_ic50,regression,"IC50, mM",mae,nested,svr,direct,14.545369,"{""C"": 1.0, ""epsilon"": 0.01, ""gamma"": ""scale""}",1.813607e+02,3.182756e+01,3.904205e+02,1.022882e+02,-9.568294e-02,5.504638e-01
6,regression_ic50,regression,"IC50, mM",mae,nested,dummy,direct,0.463110,"{""strategy"": ""median""}",2.067743e+02,3.726841e+01,4.256614e+02,1.030594e+02,-2.075058e-01,3.462400e-02
7,regression_ic50,regression,"IC50, mM",mae,nested,ridge,direct,1.008299,"{""alpha"": 1.0}",1.854449e+22,3.708898e+22,2.622587e+23,5.245174e+23,-3.714307e+42,7.428614e+42
8,regression_ic50,regression,"IC50, mM",mae,nested,lasso,direct,16.814441,"{""alpha"": 0.1}",1.126972e+27,2.253944e+27,1.593779e+28,3.187558e+28,-1.371747e+52,2.743495e+52


## Короткий разбор результата

In [7]:
winner = leaderboard.iloc[0]
baseline_rows = leaderboard[leaderboard['model_name'] == 'dummy']
baseline = baseline_rows.iloc[0] if not baseline_rows.empty else None

print(
    f"Победитель: {winner['model_name']} "
    f"({winner['mode']}), "
    f"основная метрика {winner['primary_metric']} = {winner[winner['primary_metric']]:.6f}."
)
if baseline is not None:
    print(
        f"Для сравнения dummy даёт {baseline[baseline['primary_metric']]:.6f} по той же метрике."
    )
print(f"Лучшие параметры: {winner['best_params_json']}")

Победитель: knn (direct), основная метрика mae = 166.819448.
Для сравнения dummy даёт 206.774338 по той же метрике.
Лучшие параметры: {"n_neighbors": 5, "p": 2, "weights": "uniform"}


## Итог
TODO: написать почему именно эта модель победила и порассуждать почему в этой задаче именно она показывает хороший результат